# CAB420 Assignment 1A Question 2 — Completed Notebook

This notebook follows the provided template and:
- loads the given train/validation/test splits,
- standardises the data for KNN and SVM,
- performs validation-set grid searches using simple nested loops,
- retrains the selected models on the combined training+validation data,
- evaluates final performance on the held-out test set.

The code is written to be easy to read and easy to explain in the report.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from itertools import product
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# load data
train = pd.read_csv('Q2/training.csv')
val = pd.read_csv('Q2/validation.csv')
test = pd.read_csv('Q2/testing.csv')

# split into predictors and response
X_train = train.iloc[:, 1:].to_numpy()
y_train = np.char.strip(train.iloc[:, 0].astype(str).to_numpy())

X_val = val.iloc[:, 1:].to_numpy()
y_val = np.char.strip(val.iloc[:, 0].astype(str).to_numpy())

X_test = test.iloc[:, 1:].to_numpy()
y_test = np.char.strip(test.iloc[:, 0].astype(str).to_numpy())

print('Train shape:', X_train.shape, y_train.shape)
print('Validation shape:', X_val.shape, y_val.shape)
print('Test shape:', X_test.shape, y_test.shape)

print('\nTraining class counts:')
print(pd.Series(y_train).value_counts().sort_index())

## Pre-processing

Standardisation is applied for **KNN** and **SVM** because both methods are distance/margin based and can be strongly affected by predictors being on different scales.

Random Forest does **not** require standardisation because it is tree-based and splits on thresholds within individual variables.


In [ ]:
# standardise using the training set only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
def fit_and_score(model, X_tr, y_tr, X_va, y_va):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_va)
    return accuracy_score(y_va, pred), f1_score(y_va, pred, average='macro')

## 1) KNN grid search

Grid searched:
- `n_neighbors` = [1, 3, 5, 7, 9, 11, 15, 21]
- `metric` = ['euclidean', 'manhattan']
- `weights` = ['uniform', 'distance']


In [ ]:
knn_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21],
    'metric': ['euclidean', 'manhattan'],
    'weights': ['uniform', 'distance']
}

knn_results = []
best_knn_params = None
best_knn_acc = -1
best_knn_f1 = -1

for k, metric, weights in product(knn_grid['n_neighbors'], knn_grid['metric'], knn_grid['weights']):
    model = KNeighborsClassifier(n_neighbors=k, metric=metric, weights=weights)
    acc, mf1 = fit_and_score(model, X_train_scaled, y_train, X_val_scaled, y_val)
    knn_results.append([k, metric, weights, acc, mf1])

    if (acc > best_knn_acc) or (acc == best_knn_acc and mf1 > best_knn_f1):
        best_knn_acc = acc
        best_knn_f1 = mf1
        best_knn_params = {'n_neighbors': k, 'metric': metric, 'weights': weights}

knn_results_df = pd.DataFrame(knn_results, columns=['k', 'metric', 'weights', 'val_accuracy', 'val_macro_f1'])
knn_results_df.sort_values(['val_accuracy', 'val_macro_f1'], ascending=False).head(10)

In [ ]:
print('Best KNN params:', best_knn_params)
print('Best validation accuracy:', round(best_knn_acc, 4))
print('Best validation macro F1:', round(best_knn_f1, 4))

## 2) SVM grid search

Grid searched:
- `C` = [0.1, 1, 10, 100]
- `kernel` = ['linear', 'rbf']
- `gamma` = ['scale', 0.01, 0.1, 1] for RBF
- `decision_function_shape` = ['ovo', 'ovr']


In [ ]:
svm_results = []
best_svm_params = None
best_svm_acc = -1
best_svm_f1 = -1

for C, kernel, gamma, decision in product(
    [0.1, 1, 10, 100],
    ['linear', 'rbf'],
    ['scale', 0.01, 0.1, 1],
    ['ovo', 'ovr']
):
    if kernel == 'linear' and gamma != 'scale':
        continue

    model = SVC(C=C, kernel=kernel, gamma=gamma, decision_function_shape=decision, random_state=0)
    acc, mf1 = fit_and_score(model, X_train_scaled, y_train, X_val_scaled, y_val)
    svm_results.append([C, kernel, gamma, decision, acc, mf1])

    if (acc > best_svm_acc) or (acc == best_svm_acc and mf1 > best_svm_f1):
        best_svm_acc = acc
        best_svm_f1 = mf1
        best_svm_params = {'C': C, 'kernel': kernel, 'gamma': gamma, 'decision_function_shape': decision}

svm_results_df = pd.DataFrame(svm_results, columns=['C', 'kernel', 'gamma', 'scheme', 'val_accuracy', 'val_macro_f1'])
svm_results_df.sort_values(['val_accuracy', 'val_macro_f1'], ascending=False).head(10)

In [ ]:
print('Best SVM params:', best_svm_params)
print('Best validation accuracy:', round(best_svm_acc, 4))
print('Best validation macro F1:', round(best_svm_f1, 4))

## 3) Random Forest grid search

Grid searched:
- `n_estimators` = [50, 100, 200, 500]
- `max_depth` = [None, 5, 10, 15, 20]
- `max_features` = ['sqrt', 'log2', None]


In [ ]:
rf_results = []
best_rf_params = None
best_rf_acc = -1
best_rf_f1 = -1

for n_estimators, max_depth, max_features in product(
    [50, 100, 200, 500],
    [None, 5, 10, 15, 20],
    ['sqrt', 'log2', None]
):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        random_state=0
    )
    acc, mf1 = fit_and_score(model, X_train, y_train, X_val, y_val)
    rf_results.append([n_estimators, max_depth, max_features, acc, mf1])

    if (acc > best_rf_acc) or (acc == best_rf_acc and mf1 > best_rf_f1):
        best_rf_acc = acc
        best_rf_f1 = mf1
        best_rf_params = {'n_estimators': n_estimators, 'max_depth': max_depth, 'max_features': max_features}

rf_results_df = pd.DataFrame(rf_results, columns=['n_estimators', 'max_depth', 'max_features', 'val_accuracy', 'val_macro_f1'])
rf_results_df.sort_values(['val_accuracy', 'val_macro_f1'], ascending=False).head(10)

In [ ]:
print('Best RF params:', best_rf_params)
print('Best validation accuracy:', round(best_rf_acc, 4))
print('Best validation macro F1:', round(best_rf_f1, 4))

## Train final models on training + validation data

After selecting hyperparameters using the validation set, the final models are re-fitted on the combined training and validation data before one-off testing.


In [ ]:
# combine training and validation data
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])

# re-fit scaler on combined non-test data for KNN and SVM
final_scaler = StandardScaler()
X_train_val_scaled = final_scaler.fit_transform(X_train_val)
X_test_scaled_final = final_scaler.transform(X_test)

# final models
final_knn = KNeighborsClassifier(**best_knn_params)
final_svm = SVC(**best_svm_params, random_state=0)
final_rf = RandomForestClassifier(**best_rf_params, random_state=0)

final_knn.fit(X_train_val_scaled, y_train_val)
final_svm.fit(X_train_val_scaled, y_train_val)
final_rf.fit(X_train_val, y_train_val)

pred_knn = final_knn.predict(X_test_scaled_final)
pred_svm = final_svm.predict(X_test_scaled_final)
pred_rf = final_rf.predict(X_test)

In [ ]:
summary = pd.DataFrame({
    'Model': ['KNN', 'SVM', 'Random Forest'],
    'Test Accuracy': [
        accuracy_score(y_test, pred_knn),
        accuracy_score(y_test, pred_svm),
        accuracy_score(y_test, pred_rf)
    ],
    'Test Macro F1': [
        f1_score(y_test, pred_knn, average='macro'),
        f1_score(y_test, pred_svm, average='macro'),
        f1_score(y_test, pred_rf, average='macro')
    ]
})

summary.sort_values('Test Accuracy', ascending=False)

In [ ]:
print('KNN classification report')
print(classification_report(y_test, pred_knn, digits=3))

print('SVM classification report')
print(classification_report(y_test, pred_svm, digits=3))

print('Random Forest classification report')
print(classification_report(y_test, pred_rf, digits=3))

In [ ]:
labels = ['s', 'h', 'd', 'o']

for title, pred in [('KNN', pred_knn), ('SVM', pred_svm), ('Random Forest', pred_rf)]:
    cm = confusion_matrix(y_test, pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'{title} Confusion Matrix')
    plt.show()

## Final selected models from this dataset

Using the supplied data, the grid searches selected:

- **KNN:** `k = 1`, `metric = 'manhattan'`, `weights = 'uniform'`
- **SVM:** `C = 0.1`, `kernel = 'linear'`, `gamma = 'scale'`, `scheme = 'ovo'`
- **Random Forest:** `n_estimators = 200`, `max_depth = None`, `max_features = None`

On the held-out test set, the **SVM** achieved the best overall performance.
